In [1]:
# #!/usr/bin/env python3
# """
# Semi-vectorial guided-mode solver for a high-index-contrast dielectric waveguide.

# - Geometry: 220 nm thick, 450 nm wide Si core embedded in SiO2 (all sides).
# - Finds first few guided modes at given wavelength and estimates neff.
# - Plots TE-like0 and TM-like0 vectorial E-field (approx reconstruction).
# - Sweeps wavelength and plots neff(lambda) for first 5 modes.

# Dependencies: numpy, scipy, matplotlib
# """

# from __future__ import annotations
# import numpy as np
# import scipy.sparse as sp
# import scipy.sparse.linalg as spla
# import matplotlib.pyplot as plt


# # --------------------------
# # Materials (simple dispersion option)
# # --------------------------
# def n_Si(lambda_um: float) -> float:
#     """
#     Silicon refractive index around 1.3–1.6 µm.
#     For a challenge, a constant is often acceptable; dispersion is weak here.
#     """
#     # Typical values: ~3.48 at 1.55 µm, ~3.50 at 1.31 µm (ballpark).
#     # Linear interpolation for mild dispersion:
#     lam = lambda_um
#     return 3.50 + (3.48 - 3.50) * (lam - 1.31) / (1.55 - 1.31)


# def n_SiO2(lambda_um: float) -> float:
#     """
#     Silica refractive index around 1.3–1.6 µm (weak dispersion).
#     """
#     lam = lambda_um
#     return 1.445 + (1.444 - 1.445) * (lam - 1.31) / (1.55 - 1.31)


# # --------------------------
# # Grid + geometry
# # --------------------------
# def make_grid(
#     dx_nm: float = 10.0,
#     pad_um: float = 1.5,
#     core_w_nm: float = 450.0,
#     core_t_nm: float = 220.0,
# ):
#     """
#     Returns x, y coordinate arrays (meters), and geometry extents.

#     Convention:
#       x = vertical (thickness direction)
#       y = horizontal (width direction)
#     """
#     dx = dx_nm * 1e-9
#     pad = pad_um * 1e-6

#     core_w = core_w_nm * 1e-9
#     core_t = core_t_nm * 1e-9

#     # Domain size: core + padding on all sides
#     Lx = core_t + 2 * pad
#     Ly = core_w + 2 * pad

#     Nx = int(np.round(Lx / dx)) + 1
#     Ny = int(np.round(Ly / dx)) + 1

#     x = (np.arange(Nx) - (Nx - 1) / 2) * dx
#     y = (np.arange(Ny) - (Ny - 1) / 2) * dx
#     return x, y, dx, core_w, core_t


# def eps_r_map(
#     x: np.ndarray,
#     y: np.ndarray,
#     lambda_um: float,
#     core_w_m: float,
#     core_t_m: float,
# ):
#     """
#     Relative permittivity map eps_r(x,y) for Si core in SiO2 cladding.
#     Core is centered at (0,0), thickness in x, width in y.
#     """
#     n_core = n_Si(lambda_um)
#     n_clad = n_SiO2(lambda_um)

#     eps = (n_clad 2) * np.ones((x.size, y.size), dtype=float)

#     # Core mask: |x| <= t/2 and |y| <= w/2
#     X, Y = np.meshgrid(x, y, indexing="ij")
#     core = (np.abs(X) <= core_t_m / 2) & (np.abs(Y) <= core_w_m / 2)
#     eps[core] = n_core 2
#     return eps


# # --------------------------
# # Sparse operators (2D Laplacian + simple absorbing "PML-like" layer)
# # --------------------------
# def damping_profile(N: int, n_pml: int, strength: float = 3.0) -> np.ndarray:
#     """
#     Simple edge damping profile (not a true coordinate-stretch PML, but effective
#     at suppressing boundary reflections for eigenmode screening).

#     Returns a 1D array g[i] >= 0, with larger values near edges.
#     """
#     g = np.zeros(N, dtype=float)
#     for i in range(n_pml):
#         s = (n_pml - i) / n_pml
#         g[i] = strength * (s 2)
#         g[N - 1 - i] = strength * (s 2)
#     return g


# def build_helmholtz_operator(
#     eps_r: np.ndarray,
#     k0: float,
#     dx: float,
#     n_pml: int = 30,
#     pml_strength: float = 3.0,
# ):
#     """
#     Build sparse operator A for scalar eigenproblem:
#         (∇_t^2 + k0^2 eps_r) ψ = β^2 ψ

#     We add a small imaginary "loss" near boundaries to emulate open space.
#     """
#     Nx, Ny = eps_r.shape
#     N = Nx * Ny

#     # 1D second-derivative matrices (Dirichlet-like boundaries by stencil truncation)
#     ex = np.ones(Nx)
#     ey = np.ones(Ny)

#     D2x = sp.diags([ex, -2 * ex, ex], offsets=[-1, 0, 1], shape=(Nx, Nx)) / (dx 2)
#     D2y = sp.diags([ey, -2 * ey, ey], offsets=[-1, 0, 1], shape=(Ny, Ny)) / (dx 2)

#     # 2D Laplacian via Kronecker sums
#     Ix = sp.eye(Nx, format="csr")
#     Iy = sp.eye(Ny, format="csr")
#     Lap = sp.kron(Iy, D2x, format="csr") + sp.kron(D2y, Ix, format="csr")

#     # Add weak absorbing loss near edges: eps -> eps*(1 - i*eta)
#     gx = damping_profile(Nx, n_pml, pml_strength)
#     gy = damping_profile(Ny, n_pml, pml_strength)
#     Gx, Gy = np.meshgrid(gx, gy, indexing="ij")
#     eta = (Gx + Gy)
#     eps_c = eps_r * (1.0 - 1j * 1e-3 * eta)  # small imag part

#     V = sp.diags((k0 2) * eps_c.reshape(-1), 0, shape=(N, N), format="csr")

#     A = Lap + V
#     return A


# # --------------------------
# # Mode solve + sorting
# # --------------------------
# def solve_modes(
#     eps_r: np.ndarray,
#     lambda_um: float,
#     dx: float,
#     nmodes: int = 6,
#     sigma_neff: float | None = None,
# ):
#     """
#     Solve (∇^2 + k0^2 eps_r) ψ = β^2 ψ for the top nmodes guided modes.

#     Returns:
#       betas (complex), neff (complex), modes (Nx,Ny,nmodes)
#     """
#     lam = lambda_um * 1e-6
#     k0 = 2 * np.pi / lam

#     A = build_helmholtz_operator(eps_r, k0, dx)

#     n_clad = n_SiO2(lambda_um)
#     if sigma_neff is None:
#         # target around slightly above cladding
#         sigma_neff = n_clad + 0.3

#     sigma = (k0 * sigma_neff) 2  # shift-invert target in β^2 space

#     # Solve eigenproblem A ψ = (β^2) ψ
#     vals, vecs = spla.eigs(A, k=nmodes, sigma=sigma, which="LM")

#     betas = np.sqrt(vals)  # principal branch
#     neff = betas / k0

#     # Sort: prefer real(neff) descending (more confined first)
#     order = np.argsort(-np.real(neff))
#     betas = betas[order]
#     neff = neff[order]
#     vecs = vecs[:, order]

#     Nx, Ny = eps_r.shape
#     modes = vecs.reshape(Nx, Ny, nmodes)
#     return betas, neff, modes


# # --------------------------
# # Approx "vectorial" E reconstruction (semi-vectorial)
# # --------------------------
# def reconstruct_E_from_scalar(
#     psi: np.ndarray,
#     beta: complex,
#     dx: float,
#     pol: str,
# ):
#     """
#     Given scalar dominant component psi (TE-like: Ey dominant, TM-like: Ex dominant),
#     build approximate (Ex, Ey, Ez) using:
#       - dominant component = psi
#       - transverse components ~ gradients / beta
#       - Ez from transverse divergence balance (very approximate)

#     This is meant for visualization, not metrology.
#     """
#     # Gradients
#     dpsidx = np.gradient(psi, dx, axis=0)
#     dpsidy = np.gradient(psi, dx, axis=1)

#     if pol.lower().startswith("te"):
#         Ey = psi
#         Ex = (1j / beta) * dpsidx
#         # crude longitudinal component (keeps scale reasonable)
#         Ez = (1j / beta) * dpsidy
#     else:
#         Ex = psi
#         Ey = (1j / beta) * dpsidy
#         Ez = (1j / beta) * dpsidx

#     # Normalize to max |E| = 1 for consistent plots
#     Emag = np.sqrt(np.abs(Ex) 2 + np.abs(Ey) 2 + np.abs(Ez) 2)
#     m = np.max(Emag)
#     if m > 0:
#         Ex, Ey, Ez = Ex / m, Ey / m, Ez / m
#     return Ex, Ey, Ez


# # --------------------------
# # Plotting helpers
# # --------------------------
# def plot_mode_vector_field(x, y, eps_r, Ex, Ey, Ez, title: str, stride: int = 6):
#     X, Y = np.meshgrid(y * 1e6, x * 1e6, indexing="xy")  # axes in µm (note swap for imshow)
#     Emag = np.sqrt(np.abs(Ex) 2 + np.abs(Ey) 2 + np.abs(Ez) 2)

#     plt.figure(figsize=(7.2, 4.8))
#     plt.imshow(
#         Emag,
#         extent=[y.min() * 1e6, y.max() * 1e6, x.min() * 1e6, x.max() * 1e6],
#         origin="lower",
#         aspect="auto",
#     )
#     plt.colorbar(label="|E| (normalized)")

#     # Overlay core outline via epsilon threshold
#     core = eps_r > (n_SiO2(1.55) 2 + 0.5)
#     plt.contour(
#         y * 1e6, x * 1e6, core.astype(float),
#         levels=[0.5], linewidths=1.0
#     )

#     # # Quiver of transverse E (Ex, Ey): plot real parts for a snapshot
#     # xs = x[::stride] * 1e6
#     # ys = y[::stride] * 1e6
#     # Exs = np.real(Ex[::stride, ::stride])
#     # Eys = np.real(Ey[::stride, ::stride])
#     # # Quiver expects grids (Y, X) matching array dims in "xy" coordinates:
#     # YY, XX = np.meshgrid(ys, xs, indexing="xy")
#     # plt.quiver(YY, XX, Eys, Exs, angles="xy", scale_units="xy", scale=0.15)

#     plt.xlabel("y (µm)")
#     plt.ylabel("x (µm)")
#     plt.title(title)
#     plt.tight_layout()


# def plot_neff_sweep(lams_um, neff_mat, title: str):
#     plt.figure(figsize=(7.2, 4.4))
#     for m in range(neff_mat.shape[1]):
#         plt.plot(lams_um, np.real(neff_mat[:, m]), label=f"mode {m}")
#     plt.xlabel("Wavelength (µm)")
#     plt.ylabel("Re(n_eff)")
#     plt.title(title)
#     plt.grid(True, alpha=0.3)
#     plt.legend()
#     plt.tight_layout()


# # --------------------------
# # Main "challenge" run
# # --------------------------
# def main():
#     # --- geometry / grid ---
#     x, y, dx, core_w, core_t = make_grid(
#         dx_nm=10.0,     # resolution (nm)
#         pad_um=1.5,     # padding around core (µm)
#         core_w_nm=450.0,
#         core_t_nm=220.0,
#     )

#     # --- part A: visualize TE0 and TM0 at 1310 and 1550 nm ---
#     for lam_um in [1.31, 1.55]:
#         eps = eps_r_map(x, y, lam_um, core_w, core_t)

#         # "TE-like" solve (dominant Ey): same scalar solve, but we will interpret it as TE-like
#         betas, neff, modes = solve_modes(eps, lam_um, dx, nmodes=6)

#         # Pick fundamental as mode 0
#         beta0 = betas[0]
#         psi0 = modes[:, :, 0]

#         Ex, Ey, Ez = reconstruct_E_from_scalar(psi0, beta0, dx, pol="TE")
#         plot_mode_vector_field(
#             x, y, eps, Ex, Ey, Ez,
#             title=f"TE-like₀ @ {int(lam_um*1000)} nm,  Re(neff)≈{np.real(neff[0]):.4f}"
#         )

#         # "TM-like" — we reuse the same operator but interpret the dominant component differently.
#         # For a slightly better separation, we can retarget sigma closer to higher confinement.
#         betas2, neff2, modes2 = solve_modes(eps, lam_um, dx, nmodes=6, sigma_neff=np.real(neff[0]) - 0.1)

#         beta_tm = betas2[0]
#         psi_tm = modes2[:, :, 0]

#         Ex2, Ey2, Ez2 = reconstruct_E_from_scalar(psi_tm, beta_tm, dx, pol="TM")
#         plot_mode_vector_field(
#             x, y, eps, Ex2, Ey2, Ez2,
#             title=f"TM-like₀ @ {int(lam_um*1000)} nm,  Re(neff)≈{np.real(neff2[0]):.4f}"
#         )

#     # --- part B: sweep neff of first 5 modes from 1300–1600 nm ---
#     lams = np.linspace(1.30, 1.60, 31)
#     nmodes = 5
#     neff_sweep = np.zeros((lams.size, nmodes), dtype=complex)

#     for i, lam_um in enumerate(lams):
#         eps = eps_r_map(x, y, lam_um, core_w, core_t)
#         betas, neff, modes = solve_modes(eps, lam_um, dx, nmodes=nmodes)
#         neff_sweep[i, :] = neff[:nmodes]

#     plot_neff_sweep(lams, neff_sweep, title="First 5 modes: effective index vs wavelength (semi-vectorial)")

#     plt.show()


# if __name__ == "__main__":
#     main()


SyntaxError: invalid decimal literal (2202237244.py, line 3)